# Pattern ④ RAGナレッジ連携（External Knowledge API）

## 概要

Difyには**外部ナレッジベース接続**機能があり、外部のベクトル検索をDifyの標準ナレッジとして統合できます。  
Databricks Vector Searchを外部ナレッジとして接続すると、**引用表示**や**Top K / スコア閾値**などDifyの標準機能がそのまま使えます。

## アーキテクチャ

```
Dify App
  │
  ├── ナレッジ（コンテキスト）
  │     │
  │     └── External Knowledge API
  │           │  POST /retrieval（Dify標準フォーマット）
  │           ▼
  │     中間API（リクエスト/レスポンス変換）
  │           │
  │           ▼
  │     Databricks Vector Search API
  │
  └── LLM → 検索結果 + 質問 → 回答生成（引用付き）
```

> **注意**: Databricks Vector Search APIのフォーマットはDifyの External Knowledge API と異なるため、  
> 間に**変換用の中間API**（Flask/FastAPI等）が必要です。

In [ ]:
%run ./_config

## Agent構築 ## Vector Search Indexの確認

In [ ]:
from databricks.vector_search.client import VectorSearchClient
import json

vsc = VectorSearchClient()
vs_index_name = f"{catalog}.{schema}.knowledge_base_vs_index"

try:
    idx = vsc.get_index(VECTOR_SEARCH_ENDPOINT_NAME, vs_index_name)
    print(f"✅ インデックス '{vs_index_name}' 確認済み")
    results = idx.similarity_search(
        query_text="SmartWatch X100のバッテリー",
        columns=["content", "doc_uri"],
        num_results=2
    )
    for doc in results.get("result", {}).get("data_array", []):
        print(f"  Score: {doc[-1]:.3f} | {doc[0][:80]}...")
except Exception as e:
    print(f"❌ {e}")

## Dify External Knowledge API の仕様

Difyが中間APIに送るリクエストと、期待するレスポンスの形式:

### リクエスト（Dify → 中間API）
```json
POST /retrieval
Authorization: Bearer <API_KEY>

{
  "knowledge_id": "vs_index_id",
  "query": "ユーザーの検索文",
  "retrieval_setting": {
    "top_k": 3,
    "score_threshold": 0.5
  }
}
```

### レスポンス（中間API → Dify）
```json
{
  "records": [
    {
      "content": "ドキュメント本文",
      "score": 0.85,
      "title": "ドキュメントタイトル",
      "metadata": {"source": "doc_uri"}
    }
  ]
}
```

### 中間APIが行う変換

| | Dify側 | Databricks側 |
|--|--------|-------------|
| 検索テキスト | `query` | `query_text` |
| 取得件数 | `retrieval_setting.top_k` | `num_results` |
| 結果形式 | `records[].content` | `data_array[][0]` |
| スコア | `records[].score` | `data_array[][-1]` |

## Dify側の設定手順（UI）

中間APIがデプロイ済みの前提で、Dify UIから以下の手順で設定します。

### 1. 外部ナレッジAPIの登録
1. 左メニュー **「ナレッジ」** をクリック
2. 上部の **「外部ナレッジAPI」** タブをクリック
3. **「外部ナレッジAPIを追加」** をクリック
4. 入力:
   - **名前**: `Databricks Vector Search`
   - **APIエンドポイント**: `http://<中間APIのホスト>:8089`
   - **API Key**: 中間APIに設定したキー
5. **保存**

### 2. 外部ナレッジベースの作成
1. **「ナレッジ」** → **「ナレッジを作成」**
2. **「外部ナレッジベースに接続」** を選択
3. 入力:
   - **外部ナレッジベースAPI**: 上で登録したAPIを選択
   - **外部ナレッジベースID**: 任意の識別子
4. **保存**

### 3. 検索設定の調整
1. 作成したナレッジを開く → 左メニュー **「設定」**
2. **トップK**: `3` / **スコア閾値**: ON → `0.3`
3. **保存**

### 4. 検索テスト
1. 左メニュー **「検索テスト」** で動作確認

### 5. アプリにナレッジを追加
1. Agent/Chatbot → **「コンテキスト」** → **「+ 追加」** → 作成したナレッジを選択
2. プロンプトに `{{#context#}}` を含めること（検索結果が自動注入される）

## パターン比較: Vector SearchをDifyで使う3つの方法

| | ② HTTP API | ③ MCP | ④ External Knowledge |
|--|-----------|-------|---------------------|
| 接続方式 | Workflow HTTPノード | MCPプラグイン | 外部ナレッジAPI |
| 追加インフラ | なし | なし | **中間APIサーバー必要** |
| ナレッジ管理 | 手動でLLMに渡す | Agentが判断 | **Difyが自動管理** |
| 引用表示 | 自前実装 | 自前実装 | **Dify標準機能** |
| Top K / 閾値 | 自前パラメータ管理 | 自前パラメータ管理 | **Dify UIで設定** |
| セットアップ | ◎ 簡単 | ○ 普通 | △ 中間API構築が必要 |

### 推奨

- **シンプルに使いたい** → ② HTTP API または ③ MCP
- **Difyの標準ナレッジ機能（引用表示等）を活用したい** → ④ External Knowledge API

---
## DSLインポート

上記をDify UIで手動設定する代わりに、DSLファイルをインポートできます。

1. Dify → **Studio** → **Import DSL File**
2. `dsl/04_RAG_knowledge.yml` を選択
3. インポート後、外部ナレッジAPI設定で中間APIのURLとキーを入力